# B2B Extraction Exploration

Back-to-back (.bin) → 频域校准向量 `cali_vec`。

**Pipeline:**
```
B2B .bin → _load_frames → _parse_iq → _sliding_correlate → FFT → cali_vec (U,) complex128
```

**Reference:** `src/io/b2b_extract.py`

## 1  Imports & project root

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io.bin_read import _load_frames, _parse_iq, BW_HZ, U, _S_MATCHED, _N_FFT
from src.io.b2b_extract import extract_cali_vec, diagnose_b2b_delay
from src.paths import RAW_CALI_DIR
import src.io.bin_read as _bin_read_mod
import src.io.b2b_extract as _b2b_mod

# ── 强制所有调用走 CPU numpy FFT，避免 GPU OOM ───────────────────────────────
def _sliding_correlate_cpu(iq: np.ndarray) -> np.ndarray:
    x = iq - iq.mean(axis=1, keepdims=True)
    ext = np.tile(x, (1, 3))
    S_f = np.fft.fft(_S_MATCHED, n=_N_FFT)
    F = np.fft.fft(ext, n=_N_FFT, axis=1) * S_f
    corr = np.fft.ifft(F, axis=1)
    return (corr[:, 2 * U - 1 : 3 * U - 1] / U).astype(np.complex64)

# b2b_extract 用 from-import，必须同时 patch 它自己的命名空间
_bin_read_mod._sliding_correlate = _sliding_correlate_cpu
_b2b_mod._sliding_correlate = _sliding_correlate_cpu

matplotlib.rcParams['font.family'] = 'Times New Roman'
matplotlib.rcParams['font.size'] = 12
%matplotlib inline

print(f"Project root : {PROJECT_ROOT}")
print(f"U={U}, BW={BW_HZ/1e6:.0f} MHz")
print("_sliding_correlate → CPU/numpy (patched in bin_read + b2b_extract)")

## 2  Select B2B file

In [ ]:
b2b_files = sorted(RAW_CALI_DIR.rglob("*.bin"))
print(f"RAW_CALI_DIR = {RAW_CALI_DIR}")
print(f"exists       = {RAW_CALI_DIR.exists()}")
print()
print(f"B2B .bin files ({len(b2b_files)} found):")
for i, f in enumerate(b2b_files):
    size_mb = f.stat().st_size / 1e6
    rel = f.relative_to(RAW_CALI_DIR)
    print(f"  [{i}] {rel}  ({size_mb:.1f} MB)")

In [ ]:
# ── 选择要分析的 B2B 文件 ────────────────────────────────────────────────────
# 改这里：填入 b2b_files 的序号，或直接指定完整路径
FILE_IDX = 0
B2B_PATH = b2b_files[FILE_IDX]

print(f"Selected: {B2B_PATH.relative_to(RAW_CALI_DIR)}")
print(f"Size    : {B2B_PATH.stat().st_size / 1e6:.1f} MB")

## 3  Load B2B frames → raw CIR

In [ ]:
frames = _load_frames(B2B_PATH)
print(f"frames.shape = {frames.shape}  (n_frames × FRAME_LEN)")
print(f"Total bytes: {frames.nbytes / 1e6:.2f} MB")

iq = _parse_iq(frames)
del frames
print(f"iq.shape    = {iq.shape}  (n_frames × U), dtype={iq.dtype}")

cir_b2b = _sliding_correlate_cpu(iq)   # CPU numpy FFT，无 GPU 依赖
del iq
n_frames = cir_b2b.shape[0]
print(f"cir_b2b.shape = {cir_b2b.shape}  (n_frames × U), dtype={cir_b2b.dtype}")

## 4  B2B diagnostics — hardware delay

In [ ]:
diag = diagnose_b2b_delay(cir_b2b)
print("B2B diagnostics:")
for k, v in diag.items():
    print(f"  {k}: {v}")

In [ ]:
delay_ns = np.arange(U) / BW_HZ * 1e9

# Mean PDP — numpy/CPU
pdp = (np.abs(cir_b2b) ** 2).mean(axis=0)
pdp_db = 10 * np.log10(pdp + 1e-30)
pdp_db -= pdp_db.max()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(delay_ns, pdp_db)
ax.axvline(diag['peak_delay_ns'], color='r', ls='--', lw=1,
           label=f"peak @ {diag['peak_delay_ns']:.0f} ns  ({diag['peak_power_db']:.1f} dB above noise)")
ax.set_xlabel('Delay (ns)')
ax.set_ylabel('Normalised PDP (dB)')
ax.set_title(f'B2B Mean PDP  ({n_frames} frames)')
ax.set_ylim([-60, 3])
ax.legend()
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

In [ ]:
# B2B waterfall — numpy/CPU
cir_pwr_db = 10 * np.log10(np.abs(cir_b2b) ** 2 + 1e-30)
vmax = cir_pwr_db.max()

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(
    cir_pwr_db,
    aspect='auto', origin='lower', cmap='viridis',
    vmin=vmax - 40, vmax=vmax,
    extent=[delay_ns[0], delay_ns[-1], 0, n_frames],
)
fig.colorbar(im, label='Power (dB)')
ax.set_xlabel('Delay (ns)')
ax.set_ylabel('Frame index')
ax.set_xlim(0, 5000)
ax.set_title('B2B CIR Waterfall')
fig.tight_layout()
plt.show()

## 5  Frame-to-frame phase stability

TCXO 频偏导致帧间相位漂移，验证相干平均是否可行。

In [ ]:
# Phase at the peak bin across frames
peak_bin = diag['peak_bin']
phase_at_peak = np.angle(cir_b2b[:, peak_bin])   # (n_frames,)
phase_unwrapped = np.unwrap(phase_at_peak)
phase_drift_per_frame = np.diff(phase_unwrapped)   # rad/frame

print(f"Peak bin           : {peak_bin}")
print(f"Mean phase drift   : {phase_drift_per_frame.mean():.4f} rad/frame")
print(f"Std  phase drift   : {phase_drift_per_frame.std():.4f} rad/frame")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(np.rad2deg(phase_unwrapped))
axes[0].set_ylabel('Phase (deg, unwrapped)')
axes[0].set_title(f'Peak-bin phase vs frame  (bin {peak_bin})')
axes[0].grid(True, alpha=0.4)

axes[1].plot(np.rad2deg(phase_drift_per_frame))
axes[1].set_xlabel('Frame index')
axes[1].set_ylabel('Phase increment (deg/frame)')
axes[1].set_title('Frame-to-frame phase drift — coherent avg NOT recommended if |drift| >> 0')
axes[1].grid(True, alpha=0.4)

fig.tight_layout()
plt.show()

## 6  Extract calibration vector

对比三种提取策略：
- **n_avg=1**：单帧，无偏但噪声大
- **mag_avg=True, n_avg=N**：幅度平均 + 帧0相位（推荐）
- **mag_avg=False, n_avg=N**：相干平均（TCXO漂移大时失真）

In [ ]:
import warnings

N_AVG = min(50, n_frames)

cali_single   = extract_cali_vec(B2B_PATH, n_avg=1)
cali_mag_avg  = extract_cali_vec(B2B_PATH, n_avg=N_AVG, mag_avg=True)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    cali_coherent = extract_cali_vec(B2B_PATH, n_avg=N_AVG, mag_avg=False)

print(f"cali_vec shape: {cali_mag_avg.shape}, dtype: {cali_mag_avg.dtype}")
print(f"N_AVG = {N_AVG}")

In [ ]:
freqs_mhz = np.fft.fftfreq(U, d=1.0 / BW_HZ) / 1e6
_s = np.argsort(freqs_mhz)   # sort for clean plot

def to_db(v):
    m = 20 * np.log10(np.abs(v) + 1e-30)
    return m - m.max()

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(freqs_mhz[_s], to_db(cali_single)[_s],   lw=0.8, label='single frame (n=1)', alpha=0.7)
axes[0].plot(freqs_mhz[_s], to_db(cali_mag_avg)[_s],  lw=1.2, label=f'mag_avg  (n={N_AVG})')
axes[0].plot(freqs_mhz[_s], to_db(cali_coherent)[_s], lw=1.0, label=f'coherent (n={N_AVG})', ls='--', alpha=0.7)
axes[0].set_ylabel('|H_sys| (dB, normalised)')
axes[0].set_title('Calibration Vector — Magnitude Comparison')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.4)

axes[1].plot(freqs_mhz[_s], np.unwrap(np.angle(cali_mag_avg))[_s],  label=f'mag_avg  (n={N_AVG})')
axes[1].plot(freqs_mhz[_s], np.unwrap(np.angle(cali_coherent))[_s], label=f'coherent (n={N_AVG})', ls='--', alpha=0.7)
axes[1].set_xlabel('Baseband frequency (MHz)')
axes[1].set_ylabel('Phase (rad, unwrapped)')
axes[1].set_title('Calibration Vector — Phase Comparison')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.4)

fig.tight_layout()
plt.show()

## 7  Noise floor vs N_AVG (mag_avg=True)

评估幅度平均数 `n_avg` 对校准向量噪声的影响。

In [ ]:
n_avg_list = [1, 5, 10, 20, 50, min(100, n_frames)]
noise_floor_db = []

for n in n_avg_list:
    cv = extract_cali_vec(B2B_PATH, n_avg=n, mag_avg=(n > 1))
    mag = np.abs(cv)
    noise = np.std(mag) / mag.mean()   # coefficient of variation
    noise_floor_db.append(20 * np.log10(noise + 1e-30))
    print(f"  n_avg={n:4d}  CV(|H|) = {20*np.log10(noise+1e-30):.1f} dB")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_avg_list, noise_floor_db, 'o-')
ax.set_xlabel('n_avg')
ax.set_ylabel('CV(|H_sys|) (dB)')
ax.set_title('Calibration Vector Noise vs Averaging Count')
ax.grid(True, alpha=0.4)
fig.tight_layout()
plt.show()

## 8  Final cali_vec summary

In [ ]:
# Use mag_avg as the production cali_vec
cali_vec = cali_mag_avg

mag_db = 20 * np.log10(np.abs(cali_vec) + 1e-30)
print("=== cali_vec summary ===")
print(f"  shape   : {cali_vec.shape}")
print(f"  dtype   : {cali_vec.dtype}")
print(f"  |H| max : {mag_db.max():.2f} dB")
print(f"  |H| min : {mag_db.min():.2f} dB")
print(f"  |H| std : {mag_db.std():.2f} dB")
print(f"  n_avg   : {N_AVG} (mag_avg=True)")
print(f"\nReady for apply_fr_calibration() in cali_explore.ipynb")